# OmniVLA

> OmniVLA-edge, the omni-modal goal-conditioned navigation policy, ported to MultiGrid (torch only).

In [ ]:
#| default_exp wm.omnivla.core

In [ ]:
#| hide
from nbdev.showdoc import *

[OmniVLA](https://omnivla-nav.github.io/) (Hirose et al., ICRA 2026) is a navigation policy
conditioned on goals of several modalities (a 2D pose, an egocentric goal image, language),
trained with modality dropout so that any combination can be given at test time. Its small
variant, **OmniVLA-edge**, encodes the last `context_size + 1` views of the robot with an
EfficientNet-b0, the goal image stacked on the current view with a second one, and the goal pose
with a linear layer; a transformer over these tokens (the missing modalities masked out) feeds
an MLP predicting the next `horizon` waypoints and the distance to the goal.

This is a port of OmniVLA-edge (`inference/model_omnivla_edge.py` of
[NHirose/OmniVLA](https://github.com/NHirose/OmniVLA)) with **torch as its only dependency**:

* the EfficientNet-b0 encoders are re-implemented (`EfficientNetB0`), with the parameter
  names and TensorFlow "same" padding of `efficientnet_pytorch`, so the released weights load
  and give the same features;
* the goal modalities are the ones MultiGrid has: the **pose** of the goal in the agent's frame
  (`goal_position`, in cells) and the **image** of the goal (`goal`, the view from the goal
  cell). OmniVLA's language goals (CLIP + FiLM) and satellite maps are left out (FindGoal's
  `mission` is the same in every episode);
* the waypoint head is replaced by a **discrete action head**: logits of the next `horizon`
  MultiGrid actions (the first one is executed, then the policy re-plans).

`OmniVLAEdge.load_upstream` initialises from the released OmniVLA-edge checkpoint (every
tensor whose name and shape match: the encoders always, the transformer with
`encoding_size=1024`). The real-world images it was trained on are far from MultiGrid's tiles,
so it is a starting point for training, not a zero-shot policy.

Each agent runs its own copy (decentralized), on its own views and goal, as a model of a
`FeedForwardPolicy`.

In [ ]:
#| export
# Adapted from OmniVLA's `inference/model_omnivla_edge.py` (https://github.com/NHirose/OmniVLA):
#
# MIT License
#
# Original Copyright (c) 2025 Moo Jin Kim, Chelsea Finn, Percy Liang.
# Modifications Copyright (c) 2025 Noriaki Hirose, Catherine Glossop, Dhruv Shah, Sergey Levine
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.
#
# EfficientNet-b0 follows efficientnet_pytorch (https://github.com/lukemelas/EfficientNet-PyTorch,
# Apache License 2.0, Copyright (c) Luke Melas-Kyriazi).

from __future__ import annotations

import math
from collections import deque
from functools import partial
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from fastcore.utils import patch

from stable_marl.envs.multigrid.core.actions import NavigationAction
from stable_marl.envs.multigrid.core.constants import Direction, State, Type

### EfficientNet-b0

`efficientnet_pytorch` pads its convolutions like TensorFlow's "same" padding, computed for
224 px inputs (`EfficientNet.from_name`): `k - s` zeros in total, the smaller half before. The
feature maps of 96 px (and any even size) inputs are padded the same way.

In [ ]:
#| export
# (repeats, kernel size, stride, expansion, input channels, output channels) of EfficientNet-b0's stages
_B0_STAGES = ((1, 3, 1, 1, 32, 16), (2, 3, 2, 6, 16, 24), (2, 5, 2, 6, 24, 40), (3, 3, 2, 6, 40, 80),
              (3, 5, 1, 6, 80, 112), (4, 5, 2, 6, 112, 192), (1, 3, 1, 6, 192, 320))

_BatchNorm = partial(nn.BatchNorm2d, momentum=0.01, eps=1e-3)   # TensorFlow's momentum 0.99


class _SameConv2d(nn.Conv2d):
    "A convolution with TensorFlow's 'same' padding of even-sized inputs."
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, **kwargs):
        super().__init__(in_channels, out_channels, kernel_size, stride, padding=0, **kwargs)
        total = max(kernel_size - stride, 0)
        self._pad = (total // 2, total - total // 2) * 2

    def forward(self, x):
        return super().forward(F.pad(x, self._pad))


class _MBConv(nn.Module):
    "EfficientNet's mobile inverted bottleneck: expansion, depthwise convolution, squeeze-excitation, projection."
    def __init__(self, kernel_size: int, stride: int, expand: int, in_channels: int, out_channels: int):
        super().__init__()
        mid = in_channels * expand
        if expand != 1:
            self._expand_conv, self._bn0 = nn.Conv2d(in_channels, mid, 1, bias=False), _BatchNorm(mid)
        self._depthwise_conv = _SameConv2d(mid, mid, kernel_size, stride, groups=mid, bias=False)
        self._bn1 = _BatchNorm(mid)
        squeezed = max(1, int(in_channels * 0.25))
        self._se_reduce, self._se_expand = nn.Conv2d(mid, squeezed, 1), nn.Conv2d(squeezed, mid, 1)
        self._project_conv, self._bn2 = nn.Conv2d(mid, out_channels, 1, bias=False), _BatchNorm(out_channels)
        self.residual = stride == 1 and in_channels == out_channels

    def forward(self, x, drop_rate: float = 0.0):
        h = F.silu(self._bn0(self._expand_conv(x))) if hasattr(self, '_expand_conv') else x
        h = F.silu(self._bn1(self._depthwise_conv(h)))
        h = h * torch.sigmoid(self._se_expand(F.silu(self._se_reduce(F.adaptive_avg_pool2d(h, 1)))))
        h = self._bn2(self._project_conv(h))
        if not self.residual:
            return h
        if self.training and drop_rate:          # drop connect: skip the block for some samples
            keep = 1 - drop_rate
            h = h * torch.floor(keep + torch.rand(len(h), 1, 1, 1, dtype=h.dtype, device=h.device)) / keep
        return h + x


class EfficientNetB0(nn.Module):
    """
    EfficientNet-b0's features, pooled to (N, 1280): `efficientnet_pytorch`'s
    `EfficientNet.from_name('efficientnet-b0', in_channels=...)` without its classifier, with
    its parameter names (so its weights load).
    """
    out_features = 1280

    def __init__(
        self,
        in_channels: int = 3, # Channels of the input images (e.g. 6: two RGB images stacked)
        drop_connect_rate: float = 0.2, # Drop connect rate of the last block (scaled down for earlier ones)
    ):
        super().__init__()
        self._conv_stem, self._bn0 = _SameConv2d(in_channels, 32, 3, 2, bias=False), _BatchNorm(32)
        blocks = []
        for repeats, k, s, e, cin, cout in _B0_STAGES:
            blocks += [_MBConv(k, s, e, cin, cout)] + [_MBConv(k, 1, e, cout, cout) for _ in range(repeats - 1)]
        self._blocks = nn.ModuleList(blocks)
        self._conv_head, self._bn1 = nn.Conv2d(320, self.out_features, 1, bias=False), _BatchNorm(self.out_features)
        self.drop_connect_rate = drop_connect_rate

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self._bn0(self._conv_stem(x)))
        for i, block in enumerate(self._blocks):
            x = block(x, self.drop_connect_rate * i / len(self._blocks))
        x = F.silu(self._bn1(self._conv_head(x)))
        return F.adaptive_avg_pool2d(x, 1).flatten(1)

### Goals in the agent's frame

OmniVLA's goal pose is `[x, y, cos(yaw), sin(yaw)]` in the robot's frame (x forward, y to the
left, yaw counter-clockwise). On the grid (x to the right, y down, `Direction` turning clockwise
with `right`), `goal_pose` gives the offset to a target cell, in cells, and the turn to the
target direction.

In [ ]:
#| export
GOAL_MODALITIES = ('pose', 'image')
_FORWARD = np.array([Direction(d).to_vec() for d in range(4)])   # (dx, dy) of each direction


def goal_pose(
    position: np.ndarray, # Cells of the agents (..., 2)
    direction: np.ndarray, # Their directions (...)
    target: np.ndarray, # Target cells (..., 2)
    target_direction: np.ndarray | int = 0, # Target directions (...) (FindGoal's goal image looks `right`, 0)
) -> np.ndarray:
    "`[forward, left, cos(yaw), sin(yaw)]` (..., 4) of the targets in the agents' frames (cells, counter-clockwise yaw)."
    forward = _FORWARD[np.asarray(direction)]
    left = np.stack([forward[..., 1], -forward[..., 0]], -1)
    offset = np.asarray(target, np.float32) - np.asarray(position, np.float32)
    yaw = np.broadcast_to(-(np.asarray(target_direction) - np.asarray(direction)) * (np.pi / 2), offset.shape[:-1])
    return np.stack([(offset * forward).sum(-1), (offset * left).sum(-1), np.cos(yaw), np.sin(yaw)], -1).astype(np.float32)

In [ ]:
#| hide
from fastcore.test import test_eq, test_close
# facing right (0): a cell 2 to the right and 1 down is 2 ahead and 1 to the right (left = -1)
test_close(goal_pose(np.array([1, 1]), 0, np.array([3, 2])), [2, -1, 1, 0])
# facing up (3): the same cell is 1 behind and 2 to the right; turning to `right` is a right turn (-90 degrees)
test_close(goal_pose(np.array([1, 1]), 3, np.array([3, 2]), 0), [-1, -2, 0, -1], eps=1e-6)
test_eq(goal_pose(np.zeros((4, 2, 2)), np.zeros((4, 2), int), np.ones((4, 2, 2))).shape, (4, 2, 4))

### The network

`OmniVLAEdge` keeps OmniVLA-edge's modules and their names: `obs_encoder` (each context view),
`goal_encoder_img` (the goal image stacked on the current view), `local_goal` (the goal pose),
the `compress_*` projections, the transformer `decoder` (positional encodings at OmniVLA's token
positions, masked mean of the visible tokens, the modality id appended before its MLP) and
`dist_predictor`. `action_head` replaces OmniVLA's waypoint predictor.

Images go in as uint8 `(..., H, W, 3)` (the info dict's `pixels` and `goal`): they are resized to
`image_size` and normalized as in OmniVLA (ImageNet statistics). `goal_mask` (B, 2) says which
of `pose` and `image` are given; at least one must be.

In [ ]:
#| export
_IMAGENET_MEAN, _IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
# OmniVLA's ids of the goal modalities its transformer sees (the index of its goal mask), appended to its MLP's input
_MODALITY_ID = {(True, False): 4, (True, True): 5, (False, True): 6}   # (pose, image) given


class _Decoder(nn.Module):
    "OmniVLA's `MultiLayerDecoder_mask3`: transformer, masked mean over the tokens, MLP."
    def __init__(self, embed_dim: int, seq_len: int, num_heads: int, num_layers: int, ff_factor: int,
                 output_layers: Sequence[int] = (256, 128, 64, 32)):
        super().__init__()
        self.positional_encoding = nn.Module()
        position = torch.arange(seq_len, dtype=torch.float32)[:, None]
        div = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pos_enc = torch.zeros(seq_len, embed_dim)
        pos_enc[:, 0::2], pos_enc[:, 1::2] = torch.sin(position * div), torch.cos(position * div)
        self.positional_encoding.register_buffer('pos_enc', pos_enc[None])
        layer = nn.TransformerEncoderLayer(embed_dim, num_heads, ff_factor * embed_dim, activation='gelu',
                                           batch_first=True, norm_first=True)
        self.sa_decoder = nn.TransformerEncoder(layer, num_layers, enable_nested_tensor=False)
        sizes = [embed_dim + 1, embed_dim, *output_layers]
        self.output_layers = nn.ModuleList(nn.Linear(a, b) for a, b in zip(sizes[:-1], sizes[1:]))

    def forward(self, tokens, positions, padding, modality_id):
        x = tokens + self.positional_encoding.pos_enc[:, positions]
        x = self.sa_decoder(x, src_key_padding_mask=padding)
        visible = (~padding).unsqueeze(-1).to(x.dtype)
        x = (x * visible).sum(1) / visible.sum(1)
        x = torch.cat([x, modality_id.to(x.dtype).unsqueeze(1)], 1)
        for layer in self.output_layers:
            x = F.relu(layer(x))
        return x


class OmniVLAEdge(nn.Module):
    "OmniVLA-edge with a discrete action head: goal-conditioned actions of one agent from its last views."
    def __init__(
        self,
        num_actions: int = len(NavigationAction), # Actions of the agent (MultiGrid's navigation: left, right, forward, done)
        context_size: int = 5, # Past views besides the current one (OmniVLA-edge: 5)
        horizon: int = 8, # Actions predicted per step (OmniVLA-edge: 8 waypoints); the first is executed
        encoding_size: int = 1024, # Width of the tokens and the transformer (OmniVLA-edge: 1024)
        num_heads: int = 4, # Attention heads (OmniVLA-edge: 4)
        num_layers: int = 4, # Transformer layers (OmniVLA-edge: 4)
        ff_factor: int = 4, # Feed-forward width of the transformer, in units of `encoding_size`
        image_size: int = 96, # Side the views and goal images are resized to (OmniVLA-edge: 96)
        dropout: float = 0.2, # Dropout of the pooled image features (EfficientNet's)
    ):
        super().__init__()
        self.config = dict(num_actions=num_actions, context_size=context_size, horizon=horizon,
                           encoding_size=encoding_size, num_heads=num_heads, num_layers=num_layers,
                           ff_factor=ff_factor, image_size=image_size, dropout=dropout)
        self.num_actions, self.context_size, self.horizon, self.image_size = num_actions, context_size, horizon, image_size
        E, features = encoding_size, EfficientNetB0.out_features
        self.obs_encoder = EfficientNetB0(3)
        self.goal_encoder_img = EfficientNetB0(6)
        self.compress_obs_enc = nn.Linear(features, E)
        self.compress_goal_enc_img = nn.Linear(features, E)
        self.local_goal = nn.Sequential(nn.Linear(4, E))
        # OmniVLA's tokens: views, pose, satellite map, goal image, language; here the map and language are left out
        self.decoder = _Decoder(E, context_size + 5, num_heads, num_layers, ff_factor)
        self.register_buffer('token_positions', torch.tensor([*range(context_size + 2), context_size + 3]), persistent=False)
        self.action_head = nn.Linear(32, horizon * num_actions)
        self.dist_predictor = nn.Sequential(nn.Linear(32, 1))
        self.feature_dropout = nn.Dropout(dropout)
        self.register_buffer('image_mean', torch.tensor(_IMAGENET_MEAN).view(3, 1, 1), persistent=False)
        self.register_buffer('image_std', torch.tensor(_IMAGENET_STD).view(3, 1, 1), persistent=False)
        # acting (see `get_action`)
        self.goal_modalities: tuple[str, ...] = GOAL_MODALITIES
        self.temperature = 0.0
        self._history: torch.Tensor | None = None

    def preprocess(self, images: torch.Tensor) -> torch.Tensor:
        "uint8 images (..., H, W, 3) -> normalized (..., 3, image_size, image_size)."
        lead = images.shape[:-3]
        x = images.reshape(-1, *images.shape[-3:]).permute(0, 3, 1, 2).float() / 255
        if x.shape[-2:] != (self.image_size, self.image_size):
            x = F.interpolate(x, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        return ((x - self.image_mean) / self.image_std).reshape(*lead, 3, self.image_size, self.image_size)

    def forward(
        self,
        obs: torch.Tensor, # Views (B, context_size + 1, H, W, 3) uint8, oldest first
        goal_pose: torch.Tensor, # Goal poses (B, 4) (see `goal_pose`)
        goal_image: torch.Tensor, # Goal images (B, H, W, 3) uint8
        goal_mask: torch.Tensor, # (B, 2) bool: whether the pose and the image are given
    ) -> tuple[torch.Tensor, torch.Tensor]:
        "Action logits (B, horizon, num_actions) and the predicted distance to the goal (B,)."
        B, T = obs.shape[:2]
        obs, goal_image = self.preprocess(obs), self.preprocess(goal_image)
        obs_tokens = self.compress_obs_enc(self.feature_dropout(self.obs_encoder(obs.flatten(0, 1)))).view(B, T, -1)
        image_token = self.goal_encoder_img(torch.cat([obs[:, -1], goal_image], 1))
        image_token = self.compress_goal_enc_img(self.feature_dropout(image_token))
        tokens = torch.cat([obs_tokens, self.local_goal(goal_pose.float()).unsqueeze(1), image_token.unsqueeze(1)], 1)
        goal_mask = goal_mask.bool()
        padding = torch.cat([torch.zeros(B, T, dtype=torch.bool, device=obs.device), ~goal_mask], 1)
        modality_id = torch.where(goal_mask[:, 0], torch.where(goal_mask[:, 1], 5, 4), 6)
        x = self.decoder(tokens, self.token_positions, padding, modality_id)
        return self.action_head(x).view(B, self.horizon, self.num_actions), self.dist_predictor(x).squeeze(-1)

### Acting

As the model of a `FeedForwardPolicy`, `get_action` reads each agent's latest view (`pixels`),
goal pose (from `position`, `direction`, `goal_position`) and goal image (`goal`) from the info
dict, and keeps the last `context_size + 1` views of every agent, restarted (filled with the
first view, as OmniVLA does) when an env resets (`_needs_flush`). `goal_modalities` chooses the
goals given to the model, and `temperature` how actions are picked (0: the most likely one).

In [ ]:
#| export
def _modality_mask(modalities: Sequence[str]) -> tuple[bool, bool]:
    unknown = set(modalities) - set(GOAL_MODALITIES)
    if unknown or not modalities:
        raise ValueError(f"goal modalities must be a non-empty subset of {GOAL_MODALITIES}, got {list(modalities)}")
    return tuple(m in modalities for m in GOAL_MODALITIES)


@patch
@torch.no_grad()
def get_action(self: OmniVLAEdge, info: dict) -> torch.Tensor:
    "Actions (num_envs, num_agents) for the info dict of a pool of envs (`(num_envs, 1, num_agents, ...)` columns)."
    frames = torch.as_tensor(info['pixels'])[:, -1]                  # (E, A, H, W, 3)
    E, A = frames.shape[:2]
    flush = info.get('_needs_flush')
    flush = torch.ones(E, dtype=torch.bool) if flush is None else torch.as_tensor(flush, dtype=torch.bool)
    if self._history is None or self._history.shape[:2] != (E, A) or self._history.device != frames.device:
        self._history, flush = frames.unsqueeze(2).repeat(1, 1, self.context_size + 1, 1, 1, 1), torch.ones(E, dtype=torch.bool)
    self._history = torch.cat([self._history[:, :, 1:], frames.unsqueeze(2)], 2)
    flush = flush.to(frames.device)
    self._history[flush] = frames[flush].unsqueeze(2).expand_as(self._history[flush])

    use_pose, use_image = _modality_mask(self.goal_modalities)
    if use_pose:
        latest = lambda k: torch.as_tensor(info[k])[:, -1].cpu().numpy()
        pose = torch.from_numpy(goal_pose(latest('position'), latest('direction'), latest('goal_position')))
    else:
        pose = torch.zeros(E, A, 4)
    goal_image = torch.as_tensor(info['goal'])[:, -1] if use_image else torch.zeros_like(frames)
    mask = torch.tensor([use_pose, use_image]).expand(E * A, 2)
    logits, _ = self(self._history.flatten(0, 1), pose.to(frames.device).flatten(0, 1), goal_image.flatten(0, 1),
                     mask.to(frames.device))
    logits = logits[:, 0]
    if self.temperature > 0:
        action = torch.distributions.Categorical(logits=logits / self.temperature).sample()
    else:
        action = logits.argmax(-1)
    return action.view(E, A)


@patch
def reset(self: OmniVLAEdge):
    "Forget the views of the last episodes."
    self._history = None

### Saving, loading, and the released weights

In [ ]:
#| export
OMNIVLA_EDGE_FILE = 'NHirose/omnivla-edge/resolve/main/omnivla-edge.pth'


@patch
def save(self: OmniVLAEdge, path: str | Path) -> Path:
    "Save the config and the weights to `path` (e.g. `omnivla.pt`)."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'config': self.config, 'state_dict': self.state_dict()}, path)
    return path


@patch(cls_method=True)
def load(cls: OmniVLAEdge, path: str | Path, map_location: Any = 'cpu') -> OmniVLAEdge:
    "An `OmniVLAEdge` saved with `save`."
    checkpoint = torch.load(path, map_location=map_location, weights_only=True)
    model = cls(**checkpoint['config'])
    model.load_state_dict(checkpoint['state_dict'])
    return model


@patch
def load_upstream(
    self: OmniVLAEdge,
    path: str | Path | None = None, # The released `omnivla-edge.pth`; None: downloaded once to <cache_dir>/checkpoints/
    cache_dir: str | Path | None = None,
) -> tuple[list[str], list[str]]:
    """
    Initialise from the released OmniVLA-edge weights: every tensor whose name and shape match
    (the EfficientNets always; the projections, transformer and MLP with `encoding_size=1024` and
    `context_size=5`). Returns the names loaded and the names of this model left as they were.
    """
    if path is None:
        from stable_marl.data.dataset import get_cache_dir
        from stable_marl.utils.core import HF_BASE_URL
        from stable_marl.wm.utils import _download
        path = Path(get_cache_dir(cache_dir, sub_folder='checkpoints')) / 'omnivla-edge.pth'
        if not path.exists():
            _download(f'{HF_BASE_URL}/{OMNIVLA_EDGE_FILE}', path)
    upstream = torch.load(path, map_location='cpu', weights_only=True)
    own = self.state_dict()
    matching = {k: v for k, v in upstream.items() if k in own and own[k].shape == v.shape}
    self.load_state_dict(matching, strict=False)
    return sorted(matching), sorted(set(own) - set(matching))

In [ ]:
#| hide
import tempfile

net = OmniVLAEdge(context_size=2, horizon=3, encoding_size=64, num_heads=2, num_layers=1)
B = 5
obs = torch.randint(0, 256, (B, 3, 56, 56, 3), dtype=torch.uint8)
logits, dist = net(obs, torch.randn(B, 4), torch.randint(0, 256, (B, 56, 56, 3), dtype=torch.uint8),
                   torch.tensor([[1, 0], [0, 1], [1, 1], [1, 0], [1, 1]], dtype=torch.bool))
test_eq(logits.shape, (B, 3, 4)); test_eq(dist.shape, (B,))

# the masked modality does not change the output
net.eval()
pose_only = torch.tensor([[1, 0]] * B, dtype=torch.bool)
goal_a, goal_b = (torch.randint(0, 256, (B, 56, 56, 3), dtype=torch.uint8) for _ in range(2))
pose = torch.randn(B, 4)
test_close(net(obs, pose, goal_a, pose_only)[0], net(obs, pose, goal_b, pose_only)[0], eps=1e-5)

path = net.save(f'{tempfile.mkdtemp()}/omnivla.pt')
again = OmniVLAEdge.load(path).eval()
test_close(again(obs, pose, goal_a, pose_only)[0], net(obs, pose, goal_a, pose_only)[0], eps=1e-5)

### Expert labels from the recorded grids

The datasets record the grid of every step (`state`, the `(W, H, 3)` encoding of the objects), so
the actions of the shortest-path expert (`GoToGoalPolicy`) towards any cell can be recomputed
offline, whatever policy collected the data: `shortest_path_actions` follows the expert's rule
(move forward if the cell ahead is one step closer to the target, else turn right if that faces a
closer cell, else turn left) on the cells an agent can walk on, other agents included (as the
expert ignores them).

In [ ]:
#| export
_WALKABLE_TYPES = np.zeros(len(Type), bool)
_WALKABLE_TYPES[[Type(t).to_index() for t in ('empty', 'floor', 'goal', 'lava', 'agent')]] = True


def walkable_cells(state: np.ndarray) -> np.ndarray:
    "Cells (W, H) an agent can walk on, from a grid encoding `state` (W, H, 3): empty, floor, goal, lava, agents, open doors."
    types = np.asarray(state)[..., 0]
    open_door = (types == Type.door.to_index()) & (np.asarray(state)[..., 2] == State.open.to_index())
    return _WALKABLE_TYPES[types] | open_door


def _distances(walkable: np.ndarray, target: tuple[int, int]) -> np.ndarray:
    "Steps from every cell to `target` (inf: unreachable), walking on `walkable` cells."
    W, H = walkable.shape
    distance = np.full((W, H), np.inf)
    distance[target] = 0
    queue = deque([target])
    while queue:
        x, y = queue.popleft()
        for dx, dy in _FORWARD:
            nx, ny = x + dx, y + dy
            if 0 <= nx < W and 0 <= ny < H and walkable[nx, ny] and not np.isfinite(distance[nx, ny]):
                distance[nx, ny] = distance[x, y] + 1
                queue.append((nx, ny))
    return distance


def shortest_path_actions(
    walkable: np.ndarray, # Cells an agent can walk on (W, H) (see `walkable_cells`)
    position: Sequence[int], # The agent's cell
    direction: int, # Its direction
    target: Sequence[int], # The cell to reach
    distance: np.ndarray | None = None, # Steps to `target` from every cell, if already computed
) -> np.ndarray | None:
    "`GoToGoalPolicy`'s actions from `position` until it stands on `target` (empty if already there; None: unreachable)."
    distance = _distances(walkable, tuple(int(v) for v in target)) if distance is None else distance
    (x, y), d = (int(v) for v in position), int(direction)
    if not np.isfinite(distance[x, y]):
        return None
    W, H = distance.shape
    at = lambda cx, cy: distance[cx, cy] if 0 <= cx < W and 0 <= cy < H else np.inf
    actions = []
    while distance[x, y] > 0:
        closer = [k for k, (dx, dy) in enumerate(_FORWARD) if at(x + dx, y + dy) == distance[x, y] - 1]
        if d in closer:
            actions.append(NavigationAction.forward)
            x, y = x + _FORWARD[d][0], y + _FORWARD[d][1]
        elif (d + 1) % 4 in closer:
            actions.append(NavigationAction.right)
            d = (d + 1) % 4
        else:
            actions.append(NavigationAction.left)
            d = (d - 1) % 4
    return np.array(actions, np.int64)

The offline labels are the expert's actions: replaying FindGoal episodes with `GoToGoalPolicy`
(walls and clutter included), the actions recomputed from the recorded `state` match the
recorded ones at every step.

In [ ]:
#| hide
from stable_marl import World
from stable_marl.envs.multigrid import GoToGoalPolicy, MixedPolicy

kw = dict(agents=2, size=9, num_obstacles=4, n_clutter=2, min_goal_spawn_distance=3, tile_size=4, max_steps=30)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
world.set_policy(GoToGoalPolicy())
world.reset(seed=[0, 1])
checked = 0
for _ in range(30):
    info = world.infos
    actions = world.policy.get_action(info)
    for e in range(2):
        walk = walkable_cells(info['state'][e, 0])
        for a in range(2):
            if info['terminated'][e, 0, a]:
                continue
            expert = shortest_path_actions(walk, info['position'][e, 0, a], info['direction'][e, 0, a],
                                           info['goal_position'][e, 0, a])
            test_eq(int(expert[0]), int(actions[e, a])); checked += 1
    _, _, term, trunc, world.infos = world.envs.step(actions, mask=np.ones(2, bool))
    if (term | trunc).all(-1).any():
        break
assert checked > 10
test_eq(len(shortest_path_actions(np.ones((3, 3), bool), (1, 1), 0, (1, 1))), 0)
test_eq(shortest_path_actions(np.array([[1, 0, 1]] * 3, bool).T, (0, 0), 0, (2, 0)), None)

### Training data

`OmniVLADataset` turns a dataset of MultiGrid episodes (collected by any policy: the expert,
`MixedPolicy`, random actions, ...) into training samples, one per agent and step of its
trajectory (until it reaches its goal). Each sample has the agent's last `context_size + 1`
views, a goal, the labels of the next `horizon` actions (`-100` past the goal: ignored), the
number of steps to the goal, and the goal modalities given to the model (a random non-empty
subset of `modalities`: OmniVLA's modality dropout).

The goal is the episode's goal (`goal_position` and the `goal` image), or, with probability
`hindsight_prob`, a cell the agent reached up to `max_goal_steps` steps later (its `position`,
`direction` and view then): OmniVLA's (and GNM's) hindsight relabelling, which makes every
trajectory a demonstration of reaching its own future states. The labels (`labels`) are

* `'shortest_path'`: the expert's actions towards the goal (`shortest_path_actions` on the
  recorded grid), whatever the collecting policy did;
* `'collected'`: the actions in the dataset, which reach a hindsight goal (only hindsight goals
  are used then: the actions of a non-expert policy do not lead to the episode's goal).

In [ ]:
#| export
LABELS = ('shortest_path', 'collected')
IGNORE_INDEX = -100


class OmniVLADataset(torch.utils.data.Dataset):
    "Goal-conditioned samples of every agent of a MultiGrid dataset, for `OmniVLAEdge`."
    def __init__(
        self,
        dataset, # Episodes with `pixels`, `position`, `direction`, `goal`, `goal_position`, `action`, `terminated`, `state`
        context_size: int = 5, # Past views besides the current one (as the model)
        horizon: int = 8, # Actions labelled per sample (as the model)
        labels: str = 'shortest_path', # 'shortest_path' (expert relabelling) or 'collected' (the dataset's actions)
        hindsight_prob: float = 0.5, # Probability of a future state as the goal ('collected': always)
        max_goal_steps: int = 20, # Farthest future state used as a goal, in steps
        modalities: Sequence[str] = GOAL_MODALITIES, # Goal modalities sampled from (each sample: a non-empty subset)
    ):
        if labels not in LABELS:
            raise ValueError(f"labels must be one of {LABELS}, got {labels!r}")
        _modality_mask(modalities)
        self.context_size, self.horizon, self.labels = context_size, horizon, labels
        self.hindsight_prob = 1.0 if labels == 'collected' else hindsight_prob
        self.max_goal_steps = max_goal_steps
        self.subsets = [m for m in ((True, False), (False, True), (True, True))
                        if all(GOAL_MODALITIES[i] in modalities for i in range(2) if m[i])]
        self.cols = {k: dataset.get_col_data(k) for k in ('pixels', 'position', 'direction', 'goal', 'goal_position',
                                                          'action', 'terminated', 'state')}
        items, self._walkable = [], []
        for e, (start, length) in enumerate(zip(np.asarray(dataset.offsets), np.asarray(dataset.lengths))):
            self._walkable.append(walkable_cells(self.cols['state'][start]))
            for a in range(self.cols['action'].shape[1]):
                done = np.flatnonzero(self.cols['terminated'][start:start + length, a])
                end = start + (int(done[0]) if len(done) else int(length) - 1)   # last state of the trajectory
                items += [(e, start, t, end, a) for t in range(start, end)]
        self.items = np.array(items, np.int64).reshape(-1, 5)
        self._distance_cache: dict = {}

    def __len__(self) -> int:
        return len(self.items)

    def _distance(self, episode: int, target: tuple[int, int]) -> np.ndarray:
        key = (episode, target)
        if key not in self._distance_cache:
            self._distance_cache[key] = _distances(self._walkable[episode], target)
        return self._distance_cache[key]

    def _goal(self, start: int, t: int, end: int, a: int):
        "(target cell, target direction, goal image, row of the goal or None) of a sample."
        c = self.cols
        if torch.rand(()) < self.hindsight_prob:
            rows = np.arange(t + 1, min(end, t + self.max_goal_steps) + 1)
            if self.labels == 'shortest_path':     # an expert label needs a cell to move to
                rows = rows[(c['position'][rows, a] != c['position'][t, a]).any(-1)]
            if len(rows):
                g = int(rows[torch.randint(len(rows), ()).item()])
                return tuple(int(v) for v in c['position'][g, a]), int(c['direction'][g, a]), c['pixels'][g, a], g
        return tuple(int(v) for v in c['goal_position'][t, a]), 0, c['goal'][t, a], None

    def __getitem__(self, i: int) -> dict[str, torch.Tensor]:
        e, start, t, end, a = (int(v) for v in self.items[i])
        c = self.cols
        target, target_dir, image, g = self._goal(start, t, end, a)
        if self.labels == 'collected':
            actions = np.asarray(c['action'][t:g, a]).astype(np.int64)
        else:
            actions = shortest_path_actions(self._walkable[e], c['position'][t, a], c['direction'][t, a], target,
                                            distance=self._distance(e, target))
            actions = np.zeros(0, np.int64) if actions is None else actions
        labels = np.full(self.horizon, IGNORE_INDEX, np.int64)
        labels[:min(len(actions), self.horizon)] = actions[:self.horizon]
        rows = np.clip(np.arange(t - self.context_size, t + 1), start, None)   # before the reset: the first view
        return {
            'obs': torch.from_numpy(np.ascontiguousarray(c['pixels'][rows, a])),
            'goal_pose': torch.from_numpy(goal_pose(c['position'][t, a], c['direction'][t, a], np.array(target), target_dir)),
            'goal_image': torch.from_numpy(np.ascontiguousarray(image)),
            'goal_mask': torch.tensor(self.subsets[torch.randint(len(self.subsets), ()).item()]),
            'actions': torch.from_numpy(labels),
            'distance': torch.tensor(float(len(actions))),
        }

### Training

`omnivla_losses` is OmniVLA's objective with a classification head: the cross-entropy of the
`horizon` action labels plus `dist_weight` times the squared error of the predicted distance
(ViNT's and OmniVLA's `1e-2`). `train_omnivla` samples batches with replacement for `steps`
gradient steps (AdamW).

In [ ]:
#| export
def omnivla_losses(model: OmniVLAEdge, batch: dict[str, torch.Tensor], dist_weight: float = 1e-2) -> dict[str, torch.Tensor]:
    "`loss` (to minimize), `action_loss`, `dist_loss` and `accuracy` (of the first action) of a batch of `OmniVLADataset`."
    logits, distance = model(batch['obs'], batch['goal_pose'], batch['goal_image'], batch['goal_mask'])
    labels = batch['actions']
    action_loss = F.cross_entropy(logits.flatten(0, 1).float(), labels.flatten(), ignore_index=IGNORE_INDEX)
    dist_loss = F.mse_loss(distance.float(), batch['distance'].float())
    first = labels[:, 0] != IGNORE_INDEX
    accuracy = (logits[:, 0].argmax(-1) == labels[:, 0])[first].float().mean()
    return {'loss': action_loss + dist_weight * dist_loss, 'action_loss': action_loss.detach(),
            'dist_loss': dist_loss.detach(), 'accuracy': accuracy.detach()}


def train_omnivla(
    model: OmniVLAEdge,
    data: OmniVLADataset,
    steps: int = 10_000, # Gradient steps
    batch_size: int = 64,
    lr: float = 1e-4,
    weight_decay: float = 1e-2,
    dist_weight: float = 1e-2, # Weight of the distance loss
    num_workers: int = 0, # DataLoader workers
    device: str | torch.device | None = None, # None: cuda if available
    log_every: int = 100, # Steps between two `log` calls
    log: Callable[[dict], None] | None = None, # Called with the step and the mean losses since the last call
) -> OmniVLAEdge:
    "Train `model` on `data` (in place); returns it in eval mode."
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    model.to(device).train()
    sampler = torch.utils.data.RandomSampler(data, replacement=True, num_samples=steps * batch_size)
    loader = torch.utils.data.DataLoader(data, batch_size, sampler=sampler, num_workers=num_workers,
                                         pin_memory=device.type == 'cuda', drop_last=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    totals: dict[str, float] = {}
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == 'cuda'):
            losses = omnivla_losses(model, batch, dist_weight)
        optimizer.zero_grad(set_to_none=True)
        losses['loss'].backward()
        optimizer.step()
        for k, v in losses.items():
            totals[k] = totals.get(k, 0.0) + float(v)
        if log is not None and (step % log_every == 0 or step == steps):
            n = step % log_every or log_every
            log({'step': step, **{k: v / n for k, v in totals.items()}})
            totals = {}
    model.reset()
    return model.eval()

### Tests

Data collected with `MixedPolicy` (random, epsilon-expert and waypoint episodes), labelled by the
expert: a small model learns the first action of the training samples, and acts in a `World`.

In [ ]:
#| hide
from stable_marl.data import HDF5Dataset
from stable_marl.policy import FeedForwardPolicy

tmp = tempfile.mkdtemp()
kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=15)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
world.set_policy(MixedPolicy(seed=0))
world.collect(f'{tmp}/mixed.h5', episodes=6, seed=0, progress=False)
ds = HDF5Dataset(path=f'{tmp}/mixed.h5')

for labels in LABELS:
    data = OmniVLADataset(ds, context_size=2, horizon=4, labels=labels)
    sample = data[0]
    test_eq(sample['obs'].shape, (3, 28, 28, 3)); test_eq(sample['actions'].shape, (4,))
    assert sample['goal_mask'].any()
    for i in range(len(data)):                 # the labels reach the goal: as many as the distance
        s = data[i]
        test_eq(int((s['actions'] != IGNORE_INDEX).sum()), min(int(s['distance']), 4))

data = OmniVLADataset(ds, context_size=2, horizon=4)
torch.manual_seed(0)
model = OmniVLAEdge(context_size=2, horizon=4, encoding_size=64, num_heads=2, num_layers=1, image_size=32, dropout=0.0)
logs = []
train_omnivla(model, data, steps=40, batch_size=16, lr=1e-3, log_every=20, log=logs.append, device='cpu')
test_eq([r['step'] for r in logs], [20, 40])
assert logs[-1]['action_loss'] < logs[0]['action_loss']

model.goal_modalities = ('pose',)
world.set_policy(FeedForwardPolicy(model))
results = world.evaluate(episodes=2, seed=100)
test_eq(len(results['episode_successes']), 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()